# TUGAS MANDIRI

### Konteks / Skenario

Kalian direkrut sebagai data analyst paruh waktu oleh "Kopi Nusantara", jaringan kedai kopi dengan beberapa cabang. Pemilik ingin memahami performa penjualan tiap cabang, menu terlaris, dan kontribusi barista kepala di masing-masing cabang selama periode tertentu — persis seperti studi kasus yang baru dikerjakan bersama, namun dengan data dan struktur yang sedikit berbeda.

### SparkSesssion

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder\
    .appName("Tugas7_Praktikum") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("SparkSession berhasil!")

SparkSession berhasil!


### Menyiapkan Dataset

In [2]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


### A. EXTRACT 

Baca ketiga sumber data dari HDFS (bukan disk lokal) menjadi tiga Spark DataFrame.

In [6]:
# Path data di HDFS
path_cabang = "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json"
path_menu = "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv"
path_transaksi = "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv"

# Membaca data dari HDFS
df_cabang = spark.read.json(path_cabang)
df_menu = spark.read.csv(path_menu, header=True, inferSchema=True)
df_transaksi = spark.read.csv(path_transaksi, header=True, inferSchema=True)

# Verifikasi jumlah baris
print(f"Data Cabang    : {df_cabang.count()} baris")
print(f"Data Menu      : {df_menu.count()} baris")
print(f"Data Transaksi : {df_transaksi.count()} baris")

Data Cabang    : 3 baris
Data Menu      : 15 baris
Data Transaksi : 4000 baris


### B. TRANSFORM — Join & Kolom Turunan 

Gabungkan ketiga tabel, tambahkan kolom total_penjualan (qty x harga).

In [7]:
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner")\
                            .join(df_menu, on="menu_id", how="inner")
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
df_gabungan.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


### C. TRANSFORM — Window Function 

Tentukan top-2 menu terlaris (berdasarkan total_penjualan) di setiap cabang, gunakan row_number().

In [8]:
# 1. Agregasi total penjualan per kombinasi
ringkasan_menu_cabang = df_gabungan.groupBy("nama_cabang","nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)
# 2. Buat spesifikasi window: partisi berdasarkan 'nama_cabang'
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

# 3. Terapkan row_number()
top2_menu = ringkasan_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)) \
                                    .filter(col("peringkat") <=2)
top2_menu.orderBy("nama_cabang", "peringkat").show()

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



### D. TRANSFORM — Spark SQL 
Menggunakan Spark SQL murni, tampilkan total penjualan per kepala_barista, diurutkan dari tertinggi.

In [9]:
df_gabungan.createOrReplaceTempView("transaksi_kopi")

kontribusi_barista = spark.sql("""
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM transaksi_kopi
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""")
kontribusi_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



### E. LOAD 

Simpan data gabungan (hasil bagian B) ke HDFS sebagai Parquet, path /user/[username]/tugas7/processed/laporan, dipartisi berdasarkan kategori_menu. Verifikasi dengan membaca kembali dan menghitung jumlah baris.

In [10]:
# Penyimpanan
path_tugas7_processed = "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"

# Partisi
df_gabungan.write \
    .mode("overwrite") \
    .partitionBy("kategori_menu") \
    .parquet(path_tugas7_processed)
# Verifikasi
df_verifikasi = spark.read.parquet(path_tugas7_processed)
print("Verikasi berhasil! Total baris dibaca:", df_verifikasi.count())

Verikasi berhasil! Total baris dibaca: 4000


### F. Ringkasan Eksekutif & Rekomendasi 

Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai transaksi), lalu tulis pada markdown cell (minimal 120 kata): cabang mana yang berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.


In [11]:
ringkasan_eksekutif = df_gabungan.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan_cabang"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_transaksi")
).orderBy(col("total_penjualan_cabang").desc())

ringkasan_eksekutif.show()

+--------------------+----------+--------------+----------------------+----------------+-------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan_cabang|jumlah_transaksi|rata_rata_transaksi|
+--------------------+----------+--------------+----------------------+----------------+-------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|              90542000|            1393|   64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|              87138000|            1302|  66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|              84645000|            1305|  64862.06896551724|
+--------------------+----------+--------------+----------------------+----------------+-------------------+

